# 02 – Deutsch–Jozsa algorithm

In [1]:
# Imports
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
sim = AerSimulator()
n = 3  # number of input qubits

In [2]:
# Oracles: f(x) is either constant (same output for all x) or balanced (0 for half, 1 for half).
# The last qubit is the 'answer' qubit that gets phase-kicked.
def oracle(kind):
    qc = QuantumCircuit(n + 1)
    if kind == "constant1":
        qc.x(n)                      # f(x)=1 for all x
    elif kind == "balanced":
        for i in range(n): qc.cx(i, n)  # f(x)=parity of x -> balanced
    return qc                        # 'constant0' is the empty circuit

In [3]:
# Full DJ circuit: H on inputs, answer qubit in |->, oracle, H on inputs, measure inputs
def dj(kind):
    qc = QuantumCircuit(n + 1, n)
    qc.x(n); qc.h(n)                 # answer qubit -> |->
    qc.h(range(n))                   # inputs into uniform superposition
    qc.compose(oracle(kind), inplace=True)
    qc.h(range(n))                   # interference: constant -> all zeros
    qc.measure(range(n), range(n))
    return qc

In [4]:
# Run: all-zeros '000' means constant, anything else means balanced (decided in ONE oracle call)
for kind in ["constant0", "constant1", "balanced"]:
    counts = sim.run(transpile(dj(kind), sim), shots=1000).result().get_counts()
    verdict = "constant" if counts.get("0"*n, 0) == 1000 else "balanced"
    print(kind, counts, "->", verdict)
    assert verdict == ("balanced" if kind == "balanced" else "constant")

constant0 {'000': 1000} -> constant
constant1 {'000': 1000} -> constant
balanced {'111': 1000} -> balanced
